# Chapter 9: The Maths You Actually Need
**Companion notebook** · Book 1: Python and Data Foundations for ML

Every code example from Chapter 9, in the same order as the book. Run the set-up cell first (needs an internet connection).

## Set-up: load and prepare the data

In [ ]:
!pip install ucimlrepo
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

bank_data = fetch_ucirepo(id=222)
bank = bank_data.data.features.copy()
bank["subscribed"] = bank_data.data.targets["y"]
bank = bank.fillna("unknown")
bank["subscribed_num"] = bank["subscribed"].map({"yes": 1, "no": 0})
print(bank.shape)

## 1. The centre: mean, median and mode

In [ ]:
balances = [120, 450, 300, 80, 15000]

mean = sum(balances) / len(balances)
sorted_balances = sorted(balances)
median = sorted_balances[len(sorted_balances) // 2]

print("Sorted:", sorted_balances)
print("Mean:  ", mean)
print("Median:", median)

In [ ]:
print("Mean balance:   ", round(bank["balance"].mean(), 2))
print("Median balance: ", bank["balance"].median())
print("Most common age:", bank["age"].mode()[0])

## 2. The spread: variance, standard deviation and percentiles

In [ ]:
team_a = [48, 50, 52]
team_b = [20, 50, 80]

for name, scores in [("A", team_a), ("B", team_b)]:
    mean = sum(scores) / len(scores)
    squared_gaps = [(x - mean) ** 2 for x in scores]
    variance = sum(squared_gaps) / len(scores)
    std = variance ** 0.5
    print(f"Team {name}: mean {mean}, variance {variance:.1f}, std {std:.1f}")

print("NumPy std of team B:", round(np.std(team_b), 1))

In [ ]:
q1 = bank["balance"].quantile(0.25)
q3 = bank["balance"].quantile(0.75)
iqr = q3 - q1

print(f"Q1 = {q1}, Q3 = {q3}, IQR = {iqr}")
print("Standard deviation:", round(bank["balance"].std(), 2))
print("Range:", bank["balance"].max() - bank["balance"].min())

## 3. Distributions: normal and skewed

In [ ]:
rng = np.random.default_rng(42)
heights = rng.normal(loc=170, scale=8, size=10000)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(heights, bins=40, edgecolor="white")
axes[0].set_title("Normal: 10,000 simulated heights (cm)")
axes[1].hist(bank["duration"], bins=40, edgecolor="white")
axes[1].set_title("Right-skewed: call duration (seconds)")
plt.show()

print("Heights:  mean", round(heights.mean(), 1), " median", round(np.median(heights), 1))
print("Duration: mean", round(bank["duration"].mean(), 1), " median", bank["duration"].median())
print("Skew of heights: ", round(pd.Series(heights).skew(), 2) + 0.0)
print("Skew of duration:", round(bank["duration"].skew(), 2))

In [ ]:
within_1 = np.mean(np.abs(heights - 170) <= 8)
within_2 = np.mean(np.abs(heights - 170) <= 16)
within_3 = np.mean(np.abs(heights - 170) <= 24)

print(f"Within 1 std of the mean: {within_1:.1%}")
print(f"Within 2 std of the mean: {within_2:.1%}")
print(f"Within 3 std of the mean: {within_3:.1%}")

In [ ]:
log_duration = np.log1p(bank["duration"])

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(log_duration, bins=40, edgecolor="white")
ax.set_title("Call duration after log1p: far more symmetric")
ax.set_xlabel("log(1 + duration)")
plt.show()

print("Skew after log1p:", round(log_duration.skew(), 2))

In [ ]:
age_mean = bank["age"].mean()
age_std = bank["age"].std()
bank["age_z"] = (bank["age"] - age_mean) / age_std

print(bank[["age", "age_z"]].head().round(2))
print(f"Mean of age_z: {bank['age_z'].mean():.2f}")
print(f"Std of age_z:  {bank['age_z'].std():.2f}")

## 4. Probability

In [ ]:
total = len(bank)
yes = (bank["subscribed"] == "yes").sum()
p_yes = yes / total
print(f"P(subscribe) = {yes} / {total} = {p_yes:.3f}")

retired = bank[bank["job"] == "retired"]
p_yes_given_retired = (retired["subscribed"] == "yes").mean()
print(f"P(subscribe | retired) = {p_yes_given_retired:.3f}")

p_retired = len(retired) / total
p_retired_and_yes = ((bank["job"] == "retired") & (bank["subscribed"] == "yes")).mean()
print(f"P(retired) = {p_retired:.3f}")
print(f"P(retired and subscribe) = {p_retired_and_yes:.4f}")
print(f"P(retired) x P(subscribe) = {p_retired * p_yes:.4f}")

In [ ]:
print(pd.crosstab(bank["housing"], bank["subscribed"], normalize="index").round(3))

In [ ]:
calls_planned = 1000
profit_per_yes = 60
cost_per_call = 5

expected_yes = calls_planned * p_yes
expected_profit = expected_yes * profit_per_yes - calls_planned * cost_per_call
print(f"Expected subscriptions: {expected_yes:.0f}")
print(f"Expected profit: {expected_profit:,.0f} euros")

In [ ]:
for n in [10, 100, 1000, 10000]:
    sample = bank["subscribed_num"].sample(n, random_state=1)
    print(f"{n:>6} customers: rate = {sample.mean():.3f}")

## 5. Correlation, and why it is not causation

In [ ]:
hours = [1, 2, 3, 4, 5]
score = [52, 55, 61, 64, 70]

mean_h = sum(hours) / len(hours)
mean_s = sum(score) / len(score)
together = sum((h - mean_h) * (s - mean_s) for h, s in zip(hours, score))
spread_h = sum((h - mean_h) ** 2 for h in hours) ** 0.5
spread_s = sum((s - mean_s) ** 2 for s in score) ** 0.5
r = together / (spread_h * spread_s)

print("By hand:", round(r, 3))
print("NumPy:  ", round(np.corrcoef(hours, score)[0, 1], 3))

In [ ]:
x = np.arange(-10, 11)
y = x ** 2
print("Correlation of x and x squared:", round(abs(np.corrcoef(x, y)[0, 1]), 3))
print("Correlation of age and subscribing:", round(bank["age"].corr(bank["subscribed_num"]), 3))

## 6. Vectors and matrices

In [ ]:
customer = np.array([58, 2143, 1])
weights = np.array([0.02, 0.0001, -0.8])

print("Element by element:", customer * weights)
print("Dot product:", round(customer @ weights, 4))

In [ ]:
features = bank[["age", "balance"]].head(4).copy()
features["housing"] = bank["housing"].head(4).map({"yes": 1, "no": 0})
X = features.to_numpy()

print(X)
print("Shape of X:", X.shape, " Shape of weights:", weights.shape)
scores = X @ weights
print("Scores:", scores.round(4))
print("Shape of scores:", scores.shape)

## 7. Slopes, loss and gradient descent: how a model learns

In [ ]:
calls = bank[bank["campaign"] <= 10].groupby("campaign")["subscribed_num"].mean() * 100
x = calls.index.to_numpy(dtype=float)
y = calls.to_numpy()

def predict(x, m, b):
    return m * x + b

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

print("Guess m=0,  b=10:", round(mse(y, predict(x, 0, 10)), 2))
print("Guess m=-1, b=15:", round(mse(y, predict(x, -1, 15)), 2))

In [ ]:
m, b = 0.0, 0.0
learning_rate = 0.01
history = []

for step in range(5000):
    error = predict(x, m, b) - y
    grad_m = 2 * np.mean(error * x)
    grad_b = 2 * np.mean(error)
    m = m - learning_rate * grad_m
    b = b - learning_rate * grad_b
    history.append(mse(y, predict(x, m, b)))

print(f"Learned: m = {m:.3f}, b = {b:.3f}, loss = {history[-1]:.3f}")
for step in [0, 10, 100, 1000, 4999]:
    print(f"  after step {step + 1:>4}: loss = {history[step]:.3f}")

best_m, best_b = np.polyfit(x, y, 1)
print(f"np.polyfit: m = {best_m:.3f}, b = {best_b:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history[:300])
axes[0].set_title("Loss falls as the model learns")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Mean squared error")

axes[1].scatter(x, y, label="Real rates")
axes[1].plot(x, predict(x, m, b), color="red", label=f"Learned line: y = {m:.2f}x + {b:.2f}")
axes[1].set_title("The learned line")
axes[1].set_xlabel("Calls in this campaign")
axes[1].set_ylabel("Subscription rate (%)")
axes[1].legend()
plt.show()

## 8. Is the difference real? A gentle hypothesis test

In [ ]:
rng = np.random.default_rng(42)
outcome = bank["subscribed_num"].to_numpy()
has_loan = (bank["housing"] == "yes").to_numpy()

observed = outcome[~has_loan].mean() - outcome[has_loan].mean()

fake_diffs = []
for i in range(1000):
    shuffled = rng.permutation(has_loan)
    fake_diffs.append(outcome[~shuffled].mean() - outcome[shuffled].mean())
fake_diffs = np.array(fake_diffs)

p_value = np.mean(np.abs(fake_diffs) >= abs(observed))
print(f"Observed difference: {observed:.4f}")
print(f"Largest shuffled difference: {np.abs(fake_diffs).max():.4f}")
print(f"p-value: {p_value}")

In [ ]:
from scipy.stats import chi2_contingency

table = pd.crosstab(bank["housing"], bank["subscribed"])
chi2, p, dof, expected = chi2_contingency(table)
print(table)
print(f"chi-square = {chi2:.1f}, p-value = {p:.3g}")

## Mini project: two questions from the marketing team

In [ ]:
def permutation_test(group_a, group_b, n_shuffles=1000, seed=42):
    """Return the observed difference in means (a - b) and its p-value."""
    rng = np.random.default_rng(seed)
    combined = np.concatenate([group_a, group_b])
    n_a = len(group_a)
    observed = group_a.mean() - group_b.mean()
    count = 0
    for i in range(n_shuffles):
        shuffled = rng.permutation(combined)
        diff = shuffled[:n_a].mean() - shuffled[n_a:].mean()
        if abs(diff) >= abs(observed):
            count += 1
    return observed, count / n_shuffles


def bootstrap_interval(values, n_resamples=1000, seed=42):
    """Return a 95% interval for the mean of values."""
    rng = np.random.default_rng(seed)
    means = [rng.choice(values, size=len(values), replace=True).mean() for i in range(n_resamples)]
    return np.percentile(means, 2.5), np.percentile(means, 97.5)

In [ ]:
over_60 = bank.loc[bank["age"] >= 60, "subscribed_num"].to_numpy()
under_60 = bank.loc[bank["age"] < 60, "subscribed_num"].to_numpy()

diff, p = permutation_test(over_60, under_60)
low, high = bootstrap_interval(over_60)
print(f"Question 1: over-60s ({len(over_60)} customers) vs under-60s")
print(f"  Rates: {over_60.mean():.1%} vs {under_60.mean():.1%}, difference {diff:.1%}, p-value {p}")
print(f"  95% interval for the over-60 rate: {low:.1%} to {high:.1%}")

admin = bank.loc[bank["job"] == "admin.", "subscribed_num"].to_numpy()
self_emp = bank.loc[bank["job"] == "self-employed", "subscribed_num"].to_numpy()

diff, p = permutation_test(admin, self_emp)
print(f"Question 2: admin ({len(admin)}) vs self-employed ({len(self_emp)})")
print(f"  Rates: {admin.mean():.1%} vs {self_emp.mean():.1%}, difference {diff:.1%}, p-value {p}")

## Try it yourself
Use the empty cells below for the exercises at the end of Chapter 9.

---
**Dataset credit:** S. Moro, P. Rita and P. Cortez, *Bank Marketing* dataset, UCI Machine Learning Repository (2014), https://doi.org/10.24432/C5K306. Licensed under CC BY 4.0.